# 00 — G0 pilot: CausalGym × lượng tử hoá trọng số (Pythia) — bản 2

**Mục tiêu cổng G0** (chưa phải kết quả khoa học):
1. Đo *t* = thời gian một lần học DAS 1D → ước lượng GPU-giờ.
2. Bản đồ odds của DAS trên FP theo (layer, vùng) → so bậc độ lớn với CausalGym theo đúng cách tổng hợp của bài gốc.
3. Hành vi theo cấu hình nén [bit, α] (RTN giả lập theo nhóm) → tìm vùng chuyển tiếp.
4. Chạy thử mọi đường ống của C1/C2: dùng lại hướng FP trên Q (b) có khoảng tin cậy, học lại trên Q (b') với nhiều seed, tách đoạn trước/sau layer (C2), sàn Pythia step0.

**Thay đổi so với bản 1** (sau G0 full ngày 12/11/2026, commit `1232ff0`; lý do ở sổ dự án D-08, D-09):
- Quy tắc chọn ô chính mới (`MAIN_RULE`). Quy tắc cũ chọn layer 0 tại token chủ ngữ, là ô gần như tầm thường.
- Lưu odds từng cặp. Khoảng tin cậy bootstrap lấy lại mẫu theo **cụm** (cặp gốc và cặp đảo của nó), vì dữ liệu CausalGym gồm mỗi cặp kèm bản đảo.
- Hành vi tính trên câu không trùng (bản 1 đếm mỗi câu hai lần; accuracy không đổi, nhưng n thật là 100 chứ không phải 200).
- Học lại trên Q dùng `SEEDS` seed. Cos được so theo cặp seed **khác nhau**; bản 1 dùng cùng seed 0 nên cos bị thổi phồng.
- Spearman bản đồ tính thêm trên các ô có hiệu ứng; thêm T theo từng ô.
- Bỏ các ô chắc chắn bằng 0: vùng đứng trước vùng khác nhau đầu tiên, và layer cuối ở vùng không phải vùng cuối.
- Cấu hình nén khai báo bằng danh sách `QCONFIGS`, mỗi mục là [bit, α].
- Thêm ô kiểm dữ liệu: BOS, tách token theo vùng so với tách cả câu, câu eval trùng train, tỉ lệ cặp đảo.
- Tham số bản 1 (`BITS`, `ALPHAS`, ...) bị chặn để tránh chạy nhầm lưới mặc định.

**Cách chạy mặc định:** không mở notebook này trên Kaggle. Sửa `configs/kaggle_run.yaml` → commit → push → Kaggle runner → *Save Version → Save & Run All*. Runner dùng papermill ghi đè ô `parameters` bên dưới.

**Chạy từng ô (ngoại lệ, QT-11):** import notebook này vào Kaggle, bật GPU và Internet, sửa ô `parameters`, chạy lần lượt. Xong thì Stop session.

**Kiểm code không cần GPU:** đặt `TOY_TEST = True` (model và dữ liệu giả, chạy CPU).

**Quy ước:** mốc FP = fp32; lượng tử hoá giả lập (quantize → dequantize) trên 4 ma trận Linear mỗi khối GPT-NeoX; embedding, lm head, LayerNorm giữ nguyên; W(α) = W + α·(Q(W) − W). Công thức odds giống repo gốc (`eval.py`): [logit(y_src) − logit(y_base)] sau can thiệp trừ trước can thiệp.

In [1]:
# parameters — papermill ghi đè các giá trị này từ configs/kaggle_run.yaml
MODEL = "EleutherAI/pythia-410m"
REVISION = "main"
TASK = "agr_sv_num_subj-relc"
EVAL_SPLIT = "validation"
MAX_TRAIN = 400          # số câu train (bài gốc: 400)
MAX_EVAL = 200
LAYERS = "auto"          # "auto": 5 layer cách đều; "all": mọi layer; hoặc list, ví dụ [6, 12]
REGIONS = "auto"         # "auto": vùng khác nhau đầu tiên và vùng cuối; "all": từ vùng khác nhau đầu tiên tới vùng cuối; hoặc list
MAIN_RULE = "last_early" # "last_early": vùng cuối, layer sớm nhất (trừ layer đầu và cuối) đạt MAIN_FRAC lần odds lớn nhất của vùng đó; "max": cách của bản 1
MAIN_FRAC = 0.8
MAIN_LAYER = -1          # số không âm: ép ô chính, bỏ qua MAIN_RULE
MAIN_REGION = -1         # số âm: lấy vùng cuối
QCONFIGS = [[8, 1.0], [6, 1.0], [4, 1.0], [3, 1.0], [4, 0.25], [4, 0.5], [4, 0.75]]   # mỗi mục là [bit, alpha]
RELEARN_CONFIGS = [[8, 1.0], [4, 1.0], [3, 1.0]]   # cấu hình học lại DAS trên Q ở ô chính; phải nằm trong QCONFIGS
SPLIT_CONFIGS = [[4, 1.0]]                         # cấu hình cho thí nghiệm tách đoạn C2
GROUP = 128              # group size; số không dương: lượng tử hoá theo hàng
SEEDS = 3                # số seed DAS ở ô chính, dùng cho cả FP và học lại trên Q
ODDS_MIN = 1.0           # ô có odds FP từ ngưỡng này trở lên mới vào thống kê bản đồ có hiệu ứng
N_BOOT = 2000            # số lần bootstrap
BS = 4
EVAL_BS = 16
EPOCHS = 1
LR = 5e-3
DTYPE = "float32"
SKIP_STEP0 = False
OUT_DIR = "outputs/g0_local"
REPO_DIR = "."
TOY_TEST = False         # True: model và dữ liệu giả trên CPU, chỉ để kiểm code

In [2]:
# Parameters
MODEL = "EleutherAI/pythia-410m"
TASK = "agr_sv_num_subj-relc"
MAX_TRAIN = 40
MAX_EVAL = 32
LAYERS = [6, 12, 17]
REGIONS = "all"
QCONFIGS = [[4, 1.0], [2, 0.5]]
RELEARN_CONFIGS = [[4, 1.0]]
SPLIT_CONFIGS = [[4, 1.0]]
SEEDS = 2
N_BOOT = 200
SKIP_STEP0 = True
OUT_DIR = "/kaggle/working/outputs/20261113_0506_g0b/g0b_smoke_410m"
REPO_DIR = "/tmp/repo"


## 1. Môi trường

In phiên bản thư viện và GPU (QT-26). Chặn tham số kiểu bản 1: nếu config còn `BITS`, `ALPHAS`... thì dừng ngay, tránh chạy nhầm lưới mặc định.

In [3]:
import json, math, os, random, time
from collections import Counter
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import transformers

_legacy = [k for k in ("BITS", "ALPHA_BITS", "ALPHAS", "RELEARN_BITS", "SPLIT_BITS") if k in globals()]
if _legacy:
    raise ValueError(f"Tham số bản 1 không còn dùng: {_legacy}. Đổi sang QCONFIGS / RELEARN_CONFIGS / SPLIT_CONFIGS.")

DEV = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU")
if DEV == "cpu" and not TOY_TEST:
    print("CẢNH BÁO: không thấy GPU. Kiểm tra Settings → Accelerator trên Kaggle.")

torch 2.11.0+cu130 | transformers 5.20.2 | device cuda | Tesla T4


## 2. Hàm tiện ích

- `Logger`: ghi mỗi thí nghiệm thành một dòng JSONL ngay khi xong.
- `boot_ci`: khoảng tin cậy 95% bằng bootstrap theo cụm. Mỗi lần lấy lại ngẫu nhiên các cụm (có hoàn lại), rồi tính thống kê trên mọi phần tử của các cụm được chọn.
- `ratio_ci`: T = mean(odds dùng lại trên Q) / mean(odds FP), ghép cặp theo từng cặp câu.

In [4]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class Logger:
    # Ghi mỗi thí nghiệm thành một dòng JSONL ngay khi xong, để phiên bị ngắt vẫn giữ kết quả.
    def __init__(self, path):
        self.path = path
    def __call__(self, rec):
        rec = dict(rec, time=time.strftime("%Y-%m-%d %H:%M:%S"))
        print(json.dumps({k: v for k, v in rec.items() if not isinstance(v, list)}, ensure_ascii=False), flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3:
        return float("nan")
    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1])

def boot_ci(stat, clusters, n_boot, seed=0):
    clusters = np.asarray(clusters)
    groups = [np.flatnonzero(clusters == c) for c in np.unique(clusters)]
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        pick = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        vals.append(stat(pick))
    lo, hi = np.nanpercentile(np.asarray(vals, float), [2.5, 97.5])
    return float(lo), float(hi)

def ratio_ci(num, den, clusters, n_boot):
    num, den = np.asarray(num, float), np.asarray(den, float)
    f = lambda p: num[p].mean() / den[p].mean() if den[p].mean() != 0 else np.nan
    lo, hi = boot_ci(f, clusters, n_boot)
    return float(f(np.arange(len(num)))), lo, hi

def diff_ci(a, b, clusters, n_boot):
    d = np.asarray(a, float) - np.asarray(b, float)
    lo, hi = boot_ci(lambda p: d[p].mean(), clusters, n_boot)
    return float(d.mean()), lo, hi

def same_cfg(a, b):
    return int(a[0]) == int(b[0]) and abs(float(a[1]) - float(b[1])) < 1e-9

def abs_cos(a, b):
    return abs(float(a @ b))

def mean_or_none(x):
    return float(np.mean(x)) if len(x) else None

log = Logger(os.path.join(OUT_DIR, "results.jsonl"))

## 3. Dữ liệu CausalGym

Mỗi câu là danh sách vùng (span). Tách token từng vùng rồi nối, ghi lại token cuối của mỗi vùng; can thiệp tại token cuối của vùng r.

Với `agr_sv_num_subj-relc`, template của repo gốc là `<|endoftext|>The {subject} that {verbed} the {object}`, nên 6 vùng là: 0 `<|endoftext|>The`, 1 chủ ngữ, 2 ` that`, 3 động từ trong mệnh đề, 4 ` the`, 5 tân ngữ (vị trí dự đoán động từ chính). Ô kiểm dữ liệu ở mục 7 in ra một ví dụ để xác nhận.

**Cụm:** dữ liệu gồm mỗi cặp (a, b) kèm cặp đảo (b, a) (repo gốc `data.py`, thêm *flipped pairs* vào mỗi batch). Hai cặp này chung một cụm khi bootstrap.

In [5]:
def load_rows(task, split):
    from datasets import load_dataset
    ds = load_dataset("aryaman/causalgym", split=split)
    rows = [r for r in ds if r["task"] == task]
    if not rows:
        raise ValueError(f"Không có task '{task}'. Các task có: {sorted(set(ds['task']))}")
    return rows

def encode_spans(tok, spans):
    ids, ends = [], []
    for s in spans:
        ids += tok(s, add_special_tokens=False)["input_ids"]
        ends.append(len(ids) - 1)
    return ids, ends

def build_examples(tok, rows):
    exs, multi_tok = [], 0
    for r in rows:
        b_ids, b_end = encode_spans(tok, r["base"])
        s_ids, s_end = encode_spans(tok, r["src"])
        yb_t = tok(r["base_label"], add_special_tokens=False)["input_ids"]
        ys_t = tok(r["src_label"], add_special_tokens=False)["input_ids"]
        multi_tok += int(len(yb_t) > 1) + int(len(ys_t) > 1)
        if len(b_end) != len(s_end) or yb_t[0] == ys_t[0]:
            continue
        diff = [i for i, (a, b) in enumerate(zip(r["base"], r["src"])) if a != b]
        b_txt, s_txt = "".join(r["base"]), "".join(r["src"])
        exs.append(dict(b_ids=b_ids, s_ids=s_ids, b_end=b_end, s_end=s_end, yb=yb_t[0], ys=ys_t[0], diff=diff,
                        b_txt=b_txt, s_txt=s_txt, b_spans=list(r["base"]), pair=tuple(sorted((b_txt, s_txt)))))
    return exs, multi_tok

def cluster_ids(exs):
    # Cụm = cặp không thứ tự {base, src}: cặp gốc và cặp đảo chung một cụm.
    keys = {}
    return [keys.setdefault(e["pair"], len(keys)) for e in exs]

def behavior_items(exs, clusters):
    # Mỗi câu (base hoặc src) chỉ lấy một lần, kèm nhãn đúng, nhãn sai và cụm.
    seen, items = set(), []
    for e, c in zip(exs, clusters):
        for ids, ok, bad in ((e["b_ids"], e["yb"], e["ys"]), (e["s_ids"], e["ys"], e["yb"])):
            if tuple(ids) in seen:
                continue
            seen.add(tuple(ids))
            items.append(dict(ids=ids, ok=ok, bad=bad, cl=c))
    return items

def find_regions(exs):
    n_regions = Counter(len(e["b_end"]) for e in exs).most_common(1)[0][0]
    diffs = Counter(e["diff"][0] for e in exs if e["diff"])
    return n_regions, (diffs.most_common(1)[0][0] if diffs else 0)

def _pad(seqs, pad_id, dev):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    last = torch.tensor([len(s) - 1 for s in seqs])
    return ids.to(dev), mask.to(dev), last.to(dev)

def make_batch(exs, region, pad_id, dev):
    b_ids, b_mask, b_last = _pad([e["b_ids"] for e in exs], pad_id, dev)
    s_ids, s_mask, s_last = _pad([e["s_ids"] for e in exs], pad_id, dev)
    t = lambda xs: torch.tensor(xs, device=dev)
    return dict(b_ids=b_ids, b_mask=b_mask, b_last=b_last, s_ids=s_ids, s_mask=s_mask, s_last=s_last,
                b_pos=t([e["b_end"][region] for e in exs]), s_pos=t([e["s_end"][region] for e in exs]),
                yb=t([e["yb"] for e in exs]), ys=t([e["ys"] for e in exs]))

## 4. Can thiệp 1D DII và DAS

Hook vào output của `gpt_neox.layers[l]` (residual stream sau layer l). Can thiệp chỉ thay toạ độ dọc hướng đơn vị v: h ← h + ((h_src − h) · v) v. `eval_das` trả thêm odds từng cặp để tính khoảng tin cậy.

In [6]:
class Site:
    def __init__(self, model, layer):
        self.mode, self.pos, self.src, self.v, self.captured = None, None, None, None, None
        self.handle = model.gpt_neox.layers[layer].register_forward_hook(self._hook)
    def remove(self):
        self.handle.remove()
    def _hook(self, module, inputs, out):
        if self.mode is None:
            return None
        hs = out[0] if isinstance(out, tuple) else out
        idx = torch.arange(hs.shape[0], device=hs.device)
        if self.mode == "capture":
            self.captured = hs[idx, self.pos].detach()
            return None
        v = self.v.to(hs.dtype)
        hb = hs[idx, self.pos]
        coef = (self.src.to(hs.dtype) - hb) @ v
        new = hs.clone()
        new[idx, self.pos] = hb + coef[:, None] * v[None, :]
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new

def last_logits(model, ids, mask, last):
    out = model(input_ids=ids, attention_mask=mask)
    return out.logits[torch.arange(ids.shape[0], device=ids.device), last].float()

def run_intervened(model, site, batch, v):
    with torch.no_grad():
        site.mode, site.pos = "capture", batch["s_pos"]
        model(input_ids=batch["s_ids"], attention_mask=batch["s_mask"])
    site.mode, site.pos, site.src, site.v = "intervene", batch["b_pos"], site.captured, v
    try:
        return last_logits(model, batch["b_ids"], batch["b_mask"], batch["b_last"])
    finally:
        site.mode = None

def train_das(model, layer, region, exs, seed, args, dev):
    # Học một hướng đơn vị v. Trả về (v, giây, loss cuối).
    set_seed(seed)
    d = model.config.hidden_size
    w = torch.nn.Parameter(torch.randn(d, device=dev) / math.sqrt(d))
    opt = torch.optim.Adam([w], lr=args.lr)
    order = list(range(len(exs))); random.Random(seed).shuffle(order)
    batches = [order[i:i + args.bs] for i in range(0, len(order), args.bs)] * args.epochs
    total, warm = len(batches), max(1, int(0.1 * len(batches)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    site, loss = Site(model, layer), torch.tensor(float("nan"))
    sync(); t0 = time.time()
    try:
        for bi in batches:
            batch = make_batch([exs[i] for i in bi], region, args.pad_id, dev)
            logits = run_intervened(model, site, batch, w / w.norm())
            loss = F.cross_entropy(logits, batch["ys"])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    finally:
        site.remove()
    sync()
    return (w / w.norm()).detach(), time.time() - t0, float(loss.item())

@torch.no_grad()
def eval_das(model, layer, region, exs, v, args, dev):
    # Trả về (odds trung bình, IIA, danh sách odds từng cặp).
    site, odds, hits = Site(model, layer), [], []
    try:
        for i in range(0, len(exs), args.eval_bs):
            batch = make_batch(exs[i:i + args.eval_bs], region, args.pad_id, dev)
            ar = torch.arange(batch["yb"].shape[0], device=dev)
            base = last_logits(model, batch["b_ids"], batch["b_mask"], batch["b_last"])
            inter = run_intervened(model, site, batch, v)
            lo_b = base[ar, batch["ys"]] - base[ar, batch["yb"]]
            lo_i = inter[ar, batch["ys"]] - inter[ar, batch["yb"]]
            odds += (lo_i - lo_b).tolist(); hits += (lo_i > 0).float().tolist()
    finally:
        site.remove()
    return float(np.mean(odds)), float(np.mean(hits)), odds

@torch.no_grad()
def eval_behavior(model, items, args, dev):
    # Logit diff (nhãn đúng − nhãn sai) trên từng câu không trùng.
    ld = []
    for i in range(0, len(items), args.eval_bs):
        chunk = items[i:i + args.eval_bs]
        ids, mask, last = _pad([x["ids"] for x in chunk], args.pad_id, dev)
        lg = last_logits(model, ids, mask, last)
        ar = torch.arange(len(chunk), device=dev)
        ok = torch.tensor([x["ok"] for x in chunk], device=dev)
        bad = torch.tensor([x["bad"] for x in chunk], device=dev)
        ld += (lg[ar, ok] - lg[ar, bad]).tolist()
    ld = np.asarray(ld)
    return dict(acc=float((ld > 0).mean()), logit_diff=float(ld.mean()), ld_sd=float(ld.std()), n=int(len(ld))), ld

## 5. Lượng tử hoá giả lập và nạp model

In [7]:
LIN_KEYS = ("attention.query_key_value", "attention.dense", "mlp.dense_h_to_4h", "mlp.dense_4h_to_h")

def rtn_group(W, bits, group):
    # RTN bất đối xứng theo nhóm dọc chiều input.
    out_f, in_f = W.shape
    g = in_f if (group <= 0 or in_f % group) else group
    Wg = W.reshape(out_f, in_f // g, g)
    lo, hi = Wg.amin(-1, keepdim=True), Wg.amax(-1, keepdim=True)
    qmax = 2 ** bits - 1
    scale = (hi - lo).clamp(min=1e-8) / qmax
    zero = torch.round(-lo / scale)
    q = torch.clamp(torch.round(Wg / scale) + zero, 0, qmax)
    return ((q - zero) * scale).reshape(out_f, in_f)

class Quantizer:
    # Giữ bản FP của các Linear trên CPU; set() luôn khôi phục FP rồi mới lượng tử hoá.
    def __init__(self, model):
        self.model, self.orig = model, {}
        for i, layer in enumerate(model.gpt_neox.layers):
            for k in LIN_KEYS:
                self.orig[(i, k)] = layer.get_submodule(k).weight.detach().to("cpu", copy=True)
    @torch.no_grad()
    def set(self, bits=None, group=128, alpha=1.0, layers=None):
        num = den = 0.0
        for (i, k), W0 in self.orig.items():
            p = self.model.gpt_neox.layers[i].get_submodule(k).weight
            W = W0.to(p.device, torch.float32)
            if bits is not None and (layers is None or i in layers):
                Wn = W + alpha * (rtn_group(W, bits, group) - W)
                num += float((Wn - W).pow(2).sum()); den += float(W.pow(2).sum())
                W = Wn
            p.copy_(W.to(p.dtype))
        return math.sqrt(num / den) if den > 0 else 0.0   # sai số tương đối của trọng số

def load_model(name, revision, dtype, dev):
    from transformers import AutoModelForCausalLM
    m = AutoModelForCausalLM.from_pretrained(name, revision=revision)
    m = m.to(dev, dtype)          # ép dtype tường minh
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    assert hasattr(m, "gpt_neox"), "Notebook chỉ hỗ trợ Pythia / GPT-NeoX"
    return m

## 6. Nạp dữ liệu và model

Nếu `TOY_TEST = True`, ô này thay dữ liệu, tokenizer và model bằng bản giả (không tải gì). Dữ liệu giả cũng có cặp đảo, để kiểm đường bootstrap theo cụm.

Danh sách ô (layer, vùng) bỏ hai loại chắc chắn cho odds bằng 0:
- vùng đứng trước vùng khác nhau đầu tiên (base và src giống hệt tới đó);
- layer cuối ở vùng không phải vùng cuối (không còn đường tới token dự đoán).

In [8]:
if TOY_TEST:
    from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
    class ToyTok:
        pad_token_id, eos_token_id, vocab = None, 0, {}
        def __call__(self, s, add_special_tokens=False):
            ids = []
            if s.startswith("<|endoftext|>"):
                ids.append(0); s = s[len("<|endoftext|>"):]
            for w in s.split():
                self.vocab.setdefault(w, 2 + len(self.vocab) % 100)
                ids.append(self.vocab[w])
            return {"input_ids": ids or [1]}
        def convert_ids_to_tokens(self, ids):
            inv = {v: k for k, v in self.vocab.items()}
            inv[0] = "<|endoftext|>"
            return [inv.get(i, "?") for i in ids]
    def load_rows(task, split):
        rng, out = random.Random(split), []
        nouns = [(" man", " men"), (" dog", " dogs"), (" cat", " cats")]
        for _ in range(30):
            sg = rng.random() < 0.5
            n1, n2 = rng.choice(nouns), rng.choice(nouns)
            b = ["<|endoftext|>The", n1[0] if sg else n1[1], " near the car", " that sleeps"]
            s = ["<|endoftext|>The", n2[1] if sg else n2[0], " near the car", " that sleeps"]
            lb, ls = (" is", " are") if sg else (" are", " is")
            out.append(dict(base=b, src=s, base_label=lb, src_label=ls, task=task))
            out.append(dict(base=s, src=b, base_label=ls, src_label=lb, task=task))
        return out
    def load_model(name, revision, dtype, dev):
        torch.manual_seed(1 if revision == "step0" else 0)
        cfg = GPTNeoXConfig(vocab_size=120, hidden_size=64, num_hidden_layers=6, num_attention_heads=4,
                            intermediate_size=256, max_position_embeddings=64)
        m = GPTNeoXForCausalLM(cfg).to(dev, dtype).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        return m
    tok = ToyTok()
else:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)

args = SimpleNamespace(lr=LR, bs=BS, eval_bs=EVAL_BS, epochs=EPOCHS,
                       pad_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
dtype = getattr(torch, DTYPE)

tr, mt1 = build_examples(tok, load_rows(TASK, "train"))
ev, mt2 = build_examples(tok, load_rows(TASK, EVAL_SPLIT))
n_reg, diff_reg = find_regions(tr)
tr = [e for e in tr if len(e["b_end"]) == n_reg][:MAX_TRAIN]
ev = [e for e in ev if len(e["b_end"]) == n_reg][:MAX_EVAL]
ev_cl = cluster_ids(ev)
beh_items = behavior_items(ev, ev_cl)
beh_cl = [x["cl"] for x in beh_items]

model = load_model(MODEL, REVISION, dtype, DEV)
L = model.config.num_hidden_layers
last_r = n_reg - 1
if LAYERS == "auto":
    layers = sorted({int(round(x)) for x in np.linspace(0, L - 1, 5)})
elif LAYERS == "all":
    layers = list(range(L))
else:
    layers = sorted(int(x) for x in LAYERS)
if REGIONS == "auto":
    regions = sorted({diff_reg, last_r})
elif REGIONS == "all":
    regions = list(range(diff_reg, n_reg))
else:
    regions = sorted(int(x) for x in REGIONS)

def reachable(l, r):
    return r >= diff_reg and not (l == L - 1 and r != last_r)

sites = [(l, r) for l in layers for r in regions if reachable(l, r)]
skipped = [[l, r] for l in layers for r in regions if not reachable(l, r)]

log(dict(exp="meta", model=MODEL, revision=REVISION, task=TASK, toy=TOY_TEST, notebook_version=2,
         device=torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu",
         torch=torch.__version__, transformers=transformers.__version__, dtype=DTYPE,
         n_layers=L, n_train=len(tr), n_eval=len(ev), n_eval_clusters=len(set(ev_cl)),
         n_eval_sentences=len(beh_items), n_regions=n_reg, diff_region=diff_reg,
         multi_token_labels=mt1 + mt2, n_sites=len(sites), main_rule=MAIN_RULE, main_frac=MAIN_FRAC,
         seeds=SEEDS, group=GROUP, layers=layers, regions=regions, skipped=skipped,
         qconfigs=QCONFIGS, relearn_configs=RELEARN_CONFIGS, split_configs=SPLIT_CONFIGS))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.96k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/5.52M [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  911MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

{"exp": "meta", "model": "EleutherAI/pythia-410m", "revision": "main", "task": "agr_sv_num_subj-relc", "toy": false, "notebook_version": 2, "device": "Tesla T4", "torch": "2.11.0+cu130", "transformers": "5.20.2", "dtype": "float32", "n_layers": 24, "n_train": 40, "n_eval": 32, "n_eval_clusters": 16, "n_eval_sentences": 32, "n_regions": 6, "diff_region": 1, "multi_token_labels": 0, "n_sites": 15, "main_rule": "last_early", "main_frac": 0.8, "seeds": 2, "group": 128, "time": "2026-11-13 05:08:04"}


## 7. Kiểm dữ liệu

Các điểm cần đối chiếu với repo gốc:
- **BOS:** template gốc thêm `<|endoftext|>` ở đầu câu. Kỳ vọng `first_token_is_eos` = 1.0.
- **Tách token:** repo gốc tách cả câu; notebook tách từng vùng rồi nối. Kỳ vọng `tok_mismatch` = 0.
- **Trùng lặp:** repo gốc loại câu train khỏi eval. Kỳ vọng `eval_sentences_in_train` = 0.
- **Cặp đảo:** kỳ vọng `eval_swap_frac` = 1.0, nên số cụm bằng một nửa số cặp.

In [9]:
def tok_strings(ids):
    try:
        return list(tok.convert_ids_to_tokens(ids))
    except Exception:
        return [str(i) for i in ids]

allx = tr + ev
first_is_eos = float(np.mean([e["b_ids"][0] == tok.eos_token_id for e in allx]))
mismatch = int(sum(tok("".join(e["b_spans"]), add_special_tokens=False)["input_ids"] != e["b_ids"] for e in allx))
tr_txt = {e["b_txt"] for e in tr} | {e["s_txt"] for e in tr}
ev_txt = {e["b_txt"] for e in ev} | {e["s_txt"] for e in ev}
pair_set = {(e["b_txt"], e["s_txt"]) for e in ev}
swap_frac = float(np.mean([(e["s_txt"], e["b_txt"]) in pair_set for e in ev]))
e0 = ev[0]
log(dict(exp="data_check", first_token_is_eos=first_is_eos, eos_id=tok.eos_token_id,
         tok_mismatch=mismatch, n_checked=len(allx), eval_sentences_in_train=len(ev_txt & tr_txt),
         n_eval_sentences=len(ev_txt), eval_swap_frac=swap_frac, n_eval_pairs=len(ev),
         n_eval_clusters=len(set(ev_cl)), example_spans=e0["b_spans"], example_tokens=tok_strings(e0["b_ids"]),
         example_region_ends=e0["b_end"], example_labels=tok_strings([e0["yb"], e0["ys"]])))
print("Ví dụ vùng:", e0["b_spans"])
print("Token     :", tok_strings(e0["b_ids"]))
print("Token cuối mỗi vùng:", e0["b_end"], "| nhãn base/src:", tok_strings([e0["yb"], e0["ys"]]))

{"exp": "data_check", "first_token_is_eos": 1.0, "eos_id": 0, "tok_mismatch": 0, "n_checked": 72, "eval_sentences_in_train": 0, "n_eval_sentences": 32, "eval_swap_frac": 1.0, "n_eval_pairs": 32, "n_eval_clusters": 16, "time": "2026-11-13 05:08:04"}


Ví dụ vùng: ['<|endoftext|>The', ' managers', ' that', ' injured', ' the', ' pilot']
Token     : ['<|endoftext|>', 'The', 'Ġmanagers', 'Ġthat', 'Ġinjured', 'Ġthe', 'Ġpilot']
Token cuối mỗi vùng: [1, 2, 3, 4, 5, 6] | nhãn base/src: ['Ġare', 'Ġhas']


## 8. A — Hành vi FP

Logit diff trên câu không trùng (n = số câu, không phải số cặp).

In [10]:
beh_fp, ld_fp = eval_behavior(model, beh_items, args, DEV)
log(dict(exp="behavior", bits=None, alpha=0.0, **beh_fp, ld=np.round(ld_fp, 4).tolist()))

{"exp": "behavior", "bits": null, "alpha": 0.0, "acc": 0.96875, "logit_diff": 3.7266487777233124, "ld_sd": 1.5228843884976613, "n": 32, "time": "2026-11-13 05:08:07"}


## 9. B — Bản đồ DAS trên FP (seed 0)

Học DAS ở mọi ô trong `sites`, lưu hướng, odds trung bình và odds từng cặp. Cuối ô in bảng odds (hàng: layer, cột: vùng; dấu chấm là ô bỏ qua) và chỉ số tổng hợp kiểu bài gốc cho tác vụ này: lấy max theo vùng ở mỗi layer, rồi trung bình qua layer (`plot.py` của repo gốc). Chỉ số này chỉ so được với bài gốc khi `LAYERS` và `REGIONS` đều là `"all"`.

In [11]:
fp_dirs, fp_odds, fp_pairs, t_train, t_eval = {}, {}, {}, [], []
for (l, r) in sites:
    v, tt, loss = train_das(model, l, r, tr, 0, args, DEV)
    sync(); t0 = time.time()
    odds, iia, per = eval_das(model, l, r, ev, v, args, DEV)
    sync(); te = time.time() - t0
    fp_dirs[(l, r)], fp_odds[(l, r)], fp_pairs[(l, r)] = v, odds, per
    t_train.append(tt); t_eval.append(te)
    log(dict(exp="das_fp", layer=l, region=r, seed=0, odds=odds, iia=iia, final_loss=loss,
             t_train_s=round(tt, 2), t_eval_s=round(te, 2)))

print("\nOdds FP (hàng: layer, cột: vùng)")
print("layer |" + "".join(f"{'r' + str(r):>8}" for r in regions))
for l in layers:
    print(f"{l:>5} |" + "".join(f"{fp_odds[(l, r)]:8.2f}" if (l, r) in fp_odds else f"{'.':>8}" for r in regions))

per_layer_max = [max(fp_odds[(l, r)] for r in regions if (l, r) in fp_odds)
                 for l in layers if any((l, r) in fp_odds for r in regions)]
log(dict(exp="fp_map_summary", odds_maxpos_meanlayer=float(np.mean(per_layer_max)), n_layers_used=len(per_layer_max),
         comparable_to_paper=(LAYERS == "all" and REGIONS == "all")))

{"exp": "das_fp", "layer": 6, "region": 1, "seed": 0, "odds": 0.09169745445251465, "iia": 0.03125, "final_loss": 7.835780620574951, "t_train_s": 1.46, "t_eval_s": 0.22, "time": "2026-11-13 05:08:09"}


{"exp": "das_fp", "layer": 6, "region": 2, "seed": 0, "odds": 0.05068448185920715, "iia": 0.03125, "final_loss": 7.867434024810791, "t_train_s": 0.74, "t_eval_s": 0.21, "time": "2026-11-13 05:08:10"}


{"exp": "das_fp", "layer": 6, "region": 3, "seed": 0, "odds": -0.0032604634761810303, "iia": 0.03125, "final_loss": 7.870030403137207, "t_train_s": 0.74, "t_eval_s": 0.22, "time": "2026-11-13 05:08:11"}


{"exp": "das_fp", "layer": 6, "region": 4, "seed": 0, "odds": 0.0019953250885009766, "iia": 0.03125, "final_loss": 7.86839485168457, "t_train_s": 0.79, "t_eval_s": 0.22, "time": "2026-11-13 05:08:12"}


{"exp": "das_fp", "layer": 6, "region": 5, "seed": 0, "odds": 0.00036475062370300293, "iia": 0.03125, "final_loss": 7.815825462341309, "t_train_s": 0.74, "t_eval_s": 0.22, "time": "2026-11-13 05:08:13"}


{"exp": "das_fp", "layer": 12, "region": 1, "seed": 0, "odds": 0.33192700147628784, "iia": 0.03125, "final_loss": 7.64091157913208, "t_train_s": 0.68, "t_eval_s": 0.22, "time": "2026-11-13 05:08:13"}


... (6 outputs left out)

{"exp": "das_fp", "layer": 17, "region": 3, "seed": 0, "odds": 0.13957151770591736, "iia": 0.03125, "final_loss": 7.723318576812744, "t_train_s": 0.59, "t_eval_s": 0.22, "time": "2026-11-13 05:08:19"}


{"exp": "das_fp", "layer": 17, "region": 4, "seed": 0, "odds": 0.007601261138916016, "iia": 0.03125, "final_loss": 7.831977844238281, "t_train_s": 0.58, "t_eval_s": 0.22, "time": "2026-11-13 05:08:20"}


{"exp": "das_fp", "layer": 17, "region": 5, "seed": 0, "odds": 1.6481087803840637, "iia": 0.09375, "final_loss": 6.5948486328125, "t_train_s": 0.59, "t_eval_s": 0.22, "time": "2026-11-13 05:08:21"}



Odds FP (hàng: layer, cột: vùng)
layer |      r1      r2      r3      r4      r5
    6 |    0.09    0.05   -0.00    0.00    0.00
   12 |    0.33    1.06    0.03    0.02    0.11
   17 |    1.72    0.16    0.14    0.01    1.65
{"exp": "fp_map_summary", "odds_maxpos_meanlayer": 0.9569852650165558, "n_layers_used": 3, "comparable_to_paper": false, "time": "2026-11-13 05:08:21"}


## 10. Chọn ô chính

Quy tắc ghi trong `MAIN_RULE` (sổ dự án D-08, chốt trước khi chạy G0b):
- `last_early`: chỉ xét vùng cuối (vị trí dự đoán), layer 1 tới L−2; chọn layer **sớm nhất** có odds FP ≥ `MAIN_FRAC` × odds lớn nhất của vùng cuối trên các layer đó. Ý nghĩa: layer đầu tiên mà phần lớn thông tin về số của chủ ngữ đã tới vị trí dự đoán.
- `max`: ô có odds lớn nhất, trừ layer cuối (cách của bản 1, giữ để so).
- `MAIN_LAYER` không âm: ép ô chính.

In [12]:
if MAIN_LAYER >= 0:
    main = (int(MAIN_LAYER), int(MAIN_REGION) if MAIN_REGION >= 0 else last_r)
    rule = "manual"
elif MAIN_RULE == "last_early":
    cand = sorted(s for s in fp_odds if s[1] == last_r and 0 < s[0] < L - 1)
    if not cand:
        raise ValueError("Không có ô ứng viên ở vùng cuối (layer 1..L-2). Thêm layer hoặc đặt MAIN_LAYER.")
    top = max(fp_odds[s] for s in cand)
    main = next(s for s in cand if fp_odds[s] >= MAIN_FRAC * top)
    rule = f"last_early: vùng {last_r}, layer sớm nhất có odds >= {MAIN_FRAC} x {top:.3f}"
elif MAIN_RULE == "max":
    cand = [s for s in fp_odds if s[0] != L - 1] or list(fp_odds)
    main = max(cand, key=lambda s: fp_odds[s])
    rule = "max: ô có odds lớn nhất, trừ layer cuối"
else:
    raise ValueError(f"MAIN_RULE không hợp lệ: {MAIN_RULE}")

if main not in fp_dirs:
    v, tt, _ = train_das(model, main[0], main[1], tr, 0, args, DEV)
    o, _, per = eval_das(model, main[0], main[1], ev, v, args, DEV)
    fp_dirs[main], fp_odds[main], fp_pairs[main] = v, o, per
if main not in sites:
    sites.append(main)
ml, mr = main
log(dict(exp="main_site", layer=ml, region=mr, odds=fp_odds[main], rule=rule))

{"exp": "main_site", "layer": 17, "region": 5, "odds": 1.6481087803840637, "rule": "last_early: vùng 5, layer sớm nhất có odds >= 0.8 x 1.648", "time": "2026-11-13 05:08:21"}


## 11. C — Dao động theo seed ở ô chính (FP)

Học lại DAS ở ô chính với `SEEDS` seed (seed 0 lấy từ bản đồ). Ghi |cos| của mọi cặp seed khác nhau. Đây là một nửa của null hai phía; nửa Q–Q nằm ở mục 12.

In [13]:
seed_dirs, seed_odds = [fp_dirs[main]], [fp_odds[main]]
for s in range(1, SEEDS):
    v, tt, _ = train_das(model, ml, mr, tr, s, args, DEV)
    odds, iia, _ = eval_das(model, ml, mr, ev, v, args, DEV)
    seed_dirs.append(v); seed_odds.append(odds); t_train.append(tt)
    log(dict(exp="das_fp_seed", layer=ml, region=mr, seed=s, odds=odds, iia=iia, t_train_s=round(tt, 2)))

k_fp = len(seed_dirs)
cos_fp = [abs_cos(seed_dirs[i], seed_dirs[j]) for i in range(k_fp) for j in range(i + 1, k_fp)]
log(dict(exp="seed_spread", layer=ml, region=mr, k=k_fp, odds=seed_odds,
         odds_min=min(seed_odds), odds_max=max(seed_odds),
         abs_cos_min=min(cos_fp) if cos_fp else None, abs_cos_mean=mean_or_none(cos_fp), abs_cos=cos_fp))

{"exp": "das_fp_seed", "layer": 17, "region": 5, "seed": 1, "odds": 0.9365855753421783, "iia": 0.0625, "t_train_s": 0.63, "time": "2026-11-13 05:08:22"}


{"exp": "seed_spread", "layer": 17, "region": 5, "k": 2, "odds_min": 0.9365855753421783, "odds_max": 1.6481087803840637, "abs_cos_min": 0.32083195447921753, "abs_cos_mean": 0.32083195447921753, "time": "2026-11-13 05:08:22"}


## 12. D + E — Lượng tử hoá: hành vi (a), dùng lại hướng FP (b), bản đồ (d), học lại (b')

Với mỗi cấu hình [bit, α] trong `QCONFIGS`:
- (a) hành vi trên câu không trùng; chênh logit diff so với FP kèm khoảng tin cậy 95% theo cụm;
- (b) dùng lại hướng FP ở mọi ô; T ở ô chính kèm khoảng tin cậy 95% theo cụm;
- (d) Spearman trên mọi ô và trên các ô có odds FP ≥ `ODDS_MIN`; trung vị và min của T theo từng ô có hiệu ứng;
- (b') nếu cấu hình nằm trong `RELEARN_CONFIGS`: học lại DAS trên Q với `SEEDS` seed. Ghi |cos| trong Q, |cos| chéo FP–Q theo cặp seed khác nhau (để so với null), và |cos| cùng seed (chỉ để chẩn đoán).

In [14]:
quant = Quantizer(model)
main_fp = np.asarray(fp_pairs[main])
nontriv = [s for s in sites if fp_odds[s] >= ODDS_MIN]
for bits, alpha in QCONFIGS:
    bits, alpha = int(bits), float(alpha)
    rel_err = quant.set(bits, GROUP, alpha)
    beh, ld = eval_behavior(model, beh_items, args, DEV)
    d_ld, d_lo, d_hi = diff_ci(ld, ld_fp, beh_cl, N_BOOT)
    log(dict(exp="behavior", bits=bits, group=GROUP, alpha=alpha, w_rel_err=rel_err, **beh,
             d_logit_diff=d_ld, d_logit_diff_lo=d_lo, d_logit_diff_hi=d_hi, ld=np.round(ld, 4).tolist()))

    reuse, main_q = {}, None
    for s in sites:
        o, _, per = eval_das(model, s[0], s[1], ev, fp_dirs[s], args, DEV)
        reuse[s] = o
        if s == main:
            main_q = np.asarray(per)
    T, T_lo, T_hi = ratio_ci(main_q, main_fp, ev_cl, N_BOOT)
    t_cells = [reuse[s] / fp_odds[s] for s in nontriv]
    log(dict(exp="reuse_map", bits=bits, group=GROUP, alpha=alpha, w_rel_err=rel_err,
             main_odds_fp=fp_odds[main], main_odds_reuse=reuse[main],
             transfer_T=T, transfer_T_lo=T_lo, transfer_T_hi=T_hi,
             spearman_all=spearman([fp_odds[s] for s in sites], [reuse[s] for s in sites]),
             spearman_nontriv=spearman([fp_odds[s] for s in nontriv], [reuse[s] for s in nontriv]),
             n_nontriv=len(nontriv), t_cells_median=float(np.median(t_cells)) if t_cells else None,
             t_cells_min=float(np.min(t_cells)) if t_cells else None,
             main_pairs_reuse=np.round(main_q, 4).tolist(),
             map=[[s[0], s[1], fp_odds[s], reuse[s]] for s in sites]))

    if any(same_cfg([bits, alpha], c) for c in RELEARN_CONFIGS):
        q_dirs, q_odds = [], []
        for s in range(SEEDS):
            v, tt, _ = train_das(model, ml, mr, tr, s, args, DEV)
            o, _, _ = eval_das(model, ml, mr, ev, v, args, DEV)
            q_dirs.append(v); q_odds.append(o); t_train.append(tt)
        k = len(q_dirs)
        cos_q = [abs_cos(q_dirs[i], q_dirs[j]) for i in range(k) for j in range(i + 1, k)]
        cos_cross = [abs_cos(seed_dirs[i], q_dirs[j]) for i in range(k_fp) for j in range(k) if i != j]
        cos_same = [abs_cos(seed_dirs[i], q_dirs[i]) for i in range(min(k_fp, k))]
        log(dict(exp="relearn_q", bits=bits, group=GROUP, alpha=alpha, layer=ml, region=mr, k=k,
                 odds=q_odds, odds_mean=float(np.mean(q_odds)), ratio_to_fp=float(np.mean(q_odds) / np.mean(seed_odds)),
                 abs_cos_within_fp_mean=mean_or_none(cos_fp), abs_cos_within_q_mean=mean_or_none(cos_q),
                 abs_cos_cross_mean=mean_or_none(cos_cross), abs_cos_same_seed_mean=mean_or_none(cos_same),
                 abs_cos_within_q=cos_q, abs_cos_cross=cos_cross, abs_cos_same_seed=cos_same))

{"exp": "behavior", "bits": 4, "group": 128, "alpha": 1.0, "w_rel_err": 0.09804923602057167, "acc": 0.96875, "logit_diff": 3.464144393801689, "ld_sd": 1.4959401832897203, "n": 32, "d_logit_diff": -0.26250438392162323, "d_logit_diff_lo": -0.3606852501630783, "d_logit_diff_hi": -0.17258035652339454, "time": "2026-11-13 05:08:24"}


{"exp": "reuse_map", "bits": 4, "group": 128, "alpha": 1.0, "w_rel_err": 0.09804923602057167, "main_odds_fp": 1.6481087803840637, "main_odds_reuse": 1.59672112762928, "transfer_T": 0.9688202299712227, "transfer_T_lo": 0.9168431262272215, "transfer_T_hi": 1.016476559060759, "spearman_all": 0.9857142857142854, "spearman_nontriv": 1.0, "n_nontriv": 3, "t_cells_median": 0.9688202299712227, "t_cells_min": 0.9614176751602964, "time": "2026-11-13 05:08:27"}


{"exp": "relearn_q", "bits": 4, "group": 128, "alpha": 1.0, "layer": 17, "region": 5, "k": 2, "odds_mean": 1.4742254465818405, "ratio_to_fp": 1.1407348364542822, "abs_cos_within_fp_mean": 0.32083195447921753, "abs_cos_within_q_mean": 0.36230430006980896, "abs_cos_cross_mean": 0.33256301283836365, "abs_cos_same_seed_mean": 0.9781087338924408, "time": "2026-11-13 05:08:29"}


{"exp": "behavior", "bits": 2, "group": 128, "alpha": 0.5, "w_rel_err": 0.24416114560911625, "acc": 0.625, "logit_diff": 0.6427276879549026, "ld_sd": 1.916010062951318, "n": 32, "d_logit_diff": -3.0839210897684097, "d_logit_diff_lo": -3.470323049649596, "d_logit_diff_hi": -2.7009858023375273, "time": "2026-11-13 05:08:30"}


{"exp": "reuse_map", "bits": 2, "group": 128, "alpha": 0.5, "w_rel_err": 0.24416114560911625, "main_odds_fp": 1.6481087803840637, "main_odds_reuse": 0.7956255525350571, "transfer_T": 0.482750630301023, "transfer_T_lo": 0.40228420486666855, "transfer_T_hi": 0.5561935690386554, "spearman_all": 0.7357142857142855, "spearman_nontriv": -0.5, "n_nontriv": 3, "t_cells_median": 0.25415550353850364, "t_cells_min": 0.061907754486409894, "time": "2026-11-13 05:08:33"}


## 13. F — C2: tách đoạn ở ô chính

Chỉ lượng tử hoá phía mã hoá (layer ≤ ô chính: `pre`), chỉ phía đọc ra (layer > ô chính: `post`), hoặc toàn bộ (`all`). Dùng lại hướng FP; T kèm khoảng tin cậy theo cụm.

In [15]:
for bits, alpha in SPLIT_CONFIGS:
    bits, alpha = int(bits), float(alpha)
    for name, lay in [("pre", set(range(0, ml + 1))), ("post", set(range(ml + 1, L))), ("all", None)]:
        rel_err = quant.set(bits, GROUP, alpha, layers=lay)
        beh, _ = eval_behavior(model, beh_items, args, DEV)
        o, iia, per = eval_das(model, ml, mr, ev, fp_dirs[main], args, DEV)
        T, lo, hi = ratio_ci(per, main_fp, ev_cl, N_BOOT)
        log(dict(exp="split_q", part=name, bits=bits, alpha=alpha, layer=ml, region=mr, w_rel_err=rel_err,
                 acc=beh["acc"], logit_diff=beh["logit_diff"], odds_reuse=o, iia_reuse=iia, odds_fp=fp_odds[main],
                 transfer_T=T, transfer_T_lo=lo, transfer_T_hi=hi))
quant.set(None)   # khôi phục FP

{"exp": "split_q", "part": "pre", "bits": 4, "alpha": 1.0, "layer": 17, "region": 5, "w_rel_err": 0.10077195937197116, "acc": 0.96875, "logit_diff": 3.633137136697769, "odds_reuse": 1.5637129545211792, "iia_reuse": 0.03125, "odds_fp": 1.6481087803840637, "transfer_T": 0.9487923207088202, "transfer_T_lo": 0.8973062197498527, "transfer_T_hi": 1.0078777136390067, "time": "2026-11-13 05:08:34"}


{"exp": "split_q", "part": "post", "bits": 4, "alpha": 1.0, "layer": 17, "region": 5, "w_rel_err": 0.09531069173023901, "acc": 0.96875, "logit_diff": 3.5214694291353226, "odds_reuse": 1.6578113287687302, "iia_reuse": 0.1875, "odds_fp": 1.6481087803840637, "transfer_T": 1.0058870800884911, "transfer_T_lo": 0.9725914763919363, "transfer_T_hi": 1.0397910001405168, "time": "2026-11-13 05:08:34"}


{"exp": "split_q", "part": "all", "bits": 4, "alpha": 1.0, "layer": 17, "region": 5, "w_rel_err": 0.09804923602057167, "acc": 0.96875, "logit_diff": 3.464144393801689, "odds_reuse": 1.59672112762928, "iia_reuse": 0.09375, "odds_fp": 1.6481087803840637, "transfer_T": 0.9688202299712227, "transfer_T_lo": 0.9168431262272215, "transfer_T_hi": 1.016476559060759, "time": "2026-11-13 05:08:35"}


0.0

## 14. G — Sàn: DAS trên Pythia step0 (model chưa học gì) ở ô chính

In [16]:
floor_ratio = None
if not SKIP_STEP0:
    del quant, model
    if DEV == "cuda":
        torch.cuda.empty_cache()
    m0 = load_model(MODEL, "step0", dtype, DEV)
    v, tt, _ = train_das(m0, ml, mr, tr, 0, args, DEV)
    odds, iia, _ = eval_das(m0, ml, mr, ev, v, args, DEV)
    floor_ratio = odds / fp_odds[main] if fp_odds[main] else None
    log(dict(exp="floor_step0", layer=ml, region=mr, odds=odds, iia=iia, t_train_s=round(tt, 2),
             ratio_to_fp=floor_ratio))
    del m0

## 15. Thời gian, kiểm 4 quy tắc G0, file tóm tắt

In kết quả của 4 quy tắc G0 (sổ dự án §4) để đọc nhanh. Quyết định cuối do nhóm. Ghi `summary_behavior.csv`, `summary_reuse.csv`, `fp_map.csv`.

In [17]:
import pandas as pd

t_run = float(np.median(t_train)) + float(np.median(t_eval))
log(dict(exp="timing", t_train_median_s=round(float(np.median(t_train)), 2), n_train_runs=len(t_train),
         t_eval_median_s=round(float(np.median(t_eval)), 2), n_eval_runs=len(t_eval), t_per_run_s=round(t_run, 2),
         gpu_hours_3000_runs=round(3000 * t_run / 3600, 1), gpu_hours_10000_runs=round(10000 * t_run / 3600, 1)))

recs = [json.loads(x) for x in open(os.path.join(OUT_DIR, "results.jsonl"), encoding="utf-8")]
beh_df = pd.DataFrame([r for r in recs if r["exp"] == "behavior"])
beh_df = beh_df[[c for c in ["bits", "alpha", "w_rel_err", "acc", "logit_diff", "d_logit_diff",
                             "d_logit_diff_lo", "d_logit_diff_hi", "n"] if c in beh_df.columns]]
reuse_df = pd.DataFrame([r for r in recs if r["exp"] == "reuse_map"])
if len(reuse_df):
    reuse_df = reuse_df[["bits", "alpha", "w_rel_err", "main_odds_reuse", "transfer_T", "transfer_T_lo",
                         "transfer_T_hi", "spearman_all", "spearman_nontriv", "t_cells_median", "t_cells_min"]]
map_df = pd.DataFrame([dict(layer=s[0], region=s[1], odds=fp_odds[s]) for s in sorted(fp_odds)])
pd.set_option("display.width", 200)
print("\nHành vi theo cấu hình nén:\n", beh_df.to_string(index=False))
print("\nDùng lại hướng FP:\n", reuse_df.to_string(index=False))
beh_df.to_csv(os.path.join(OUT_DIR, "summary_behavior.csv"), index=False)
reuse_df.to_csv(os.path.join(OUT_DIR, "summary_reuse.csv"), index=False)
map_df.to_csv(os.path.join(OUT_DIR, "fp_map.csv"), index=False)

acc_fp = beh_fp["acc"]
print("\n=== 4 quy tắc G0 (sổ §4) ===")
print(f"1. Thời gian: t = {t_run:.2f} s ->", "giữ kế hoạch (<= 60 s)" if t_run <= 60 else "> 60 s: thu hẹp học lại")
q = beh_df.dropna(subset=["bits"]).sort_values("w_rel_err")
print("2. Lưới nén: accuracy so với FP (mục tiêu: 3-4 mức từ >= 0.95 x FP xuống gần 0.5):")
for _, r in q.iterrows():
    print(f"   {int(r.bits)} bit, alpha {r.alpha:.2f}: w_rel_err {r.w_rel_err:.3f}, acc {r.acc:.2f} ({r.acc / acc_fp:.2f} x FP)")
b8 = q[(q.bits == 8) & (q.alpha == 1.0)]
if len(b8) and float(b8.acc.iloc[0]) < 0.95 * acc_fp:
    print("   CẢNH BÁO: 8 bit đã tụt dưới 0.95 x FP -> nghi lỗi code.")
print("3. Sàn step0:", "bỏ qua" if floor_ratio is None else
      f"tỉ lệ {floor_ratio:.3f} -> " + ("(b') có thông tin (< 0.5)" if floor_ratio < 0.5 else "(b') thiếu thông tin (>= 0.5)"))
print("4. Seed:", "chưa đủ seed" if not cos_fp else
      f"|cos| min {min(cos_fp):.3f}, TB {np.mean(cos_fp):.3f} (k = {k_fp}) -> " +
      ("trục (c) dùng được (>= 0.9)" if min(cos_fp) >= 0.9 else "trục (c) không dùng được (< 0.9)"))
print("\nXong. Kết quả trong", OUT_DIR)

{"exp": "timing", "t_train_median_s": 0.65, "n_train_runs": 18, "t_eval_median_s": 0.22, "n_eval_runs": 15, "t_per_run_s": 0.87, "gpu_hours_3000_runs": 0.7, "gpu_hours_10000_runs": 2.4, "time": "2026-11-13 05:08:36"}



Hành vi theo cấu hình nén:
  bits  alpha  w_rel_err     acc  logit_diff  d_logit_diff  d_logit_diff_lo  d_logit_diff_hi  n
  NaN    0.0        NaN 0.96875    3.726649           NaN              NaN              NaN 32
  4.0    1.0   0.098049 0.96875    3.464144     -0.262504        -0.360685        -0.172580 32
  2.0    0.5   0.244161 0.62500    0.642728     -3.083921        -3.470323        -2.700986 32

Dùng lại hướng FP:
  bits  alpha  w_rel_err  main_odds_reuse  transfer_T  transfer_T_lo  transfer_T_hi  spearman_all  spearman_nontriv  t_cells_median  t_cells_min
    4    1.0   0.098049         1.596721    0.968820       0.916843       1.016477      0.985714               1.0        0.968820     0.961418
    2    0.5   0.244161         0.795626    0.482751       0.402284       0.556194      0.735714              -0.5        0.254156     0.061908

=== 4 quy tắc G0 (sổ §4) ===
1. Thời gian: t = 0.87 s -> giữ kế hoạch (<= 60 s)
2. Lưới nén: accuracy so với FP (mục tiêu: 3-4 mức từ >= 